# InfraRisk AI - Component 3: Exploratory Data Analysis (Macroeconomic & Market Data)

This notebook performs Exploratory Data Analysis (EDA) on the macroeconomic and market data used to model project credit risk.
We analyze:
- Commodity prices (Oil, Gas, Steel, Cement proxy).
- Exchange rate trends and sovereign CDS spreads.
- US Treasury yield curves and Nelson-Siegel fitted parameter paths.
- Macroeconomic indicators from WDI (Inflation, GDP growth, Government Effectiveness).

We generate 15+ publication-quality interactive Plotly visualizations.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Adjust sys.path to import src
sys.path.append(os.path.abspath('..'))
from src.data.market_data_loader import MarketDataLoader
from src.data.world_bank_loader import WorldBankLoader

# Setup directory
os.makedirs('../data/market', exist_ok=True)

## 1. Load Market & Sovereign Data

In [ ]:
# Initialize loaders and fetch data
market_loader = MarketDataLoader(cache_dir='../data/market')
if not os.path.exists('../data/market/market_data_combined.csv'):
    print("Generating new market data combined dataset...")
    df_market = market_loader.fetch_all_market_data()
else:
    print("Loading cached market dataset...")
    df_market = pd.read_csv('../data/market/market_data_combined.csv', parse_dates=['Date'])
    df_market.set_index('Date', inplace=True)

wb_loader = WorldBankLoader(cache_dir='../data')
if not os.path.exists('../data/world_bank_combined.csv'):
    df_wb = wb_loader.get_combined_dataset(10000)
else:
    df_wb = pd.read_csv('../data/world_bank_combined.csv')

print(f"Market Data shape: {df_market.shape}")
print(f"World Bank Data shape: {df_wb.shape}")

In [ ]:
df_market.head()

In [ ]:
df_market.describe()

## 2. Interactive Visualizations (15+ Plotly Visualizations)

### Viz 1: Crude Oil Price Trend (WTI Futures)

In [ ]:
fig = px.line(df_market.reset_index(), x="Date", y="Crude_Oil",
             title="Crude Oil Price Trend (WTI Futures)",
             labels={"Crude_Oil": "Price (USD/Barrel)"},
             template="plotly_white", color_discrete_sequence=["#d62728"])
fig.show()

### Viz 2: Natural Gas Price Trend (Henry Hub Futures)

In [ ]:
fig = px.line(df_market.reset_index(), x="Date", y="Natural_Gas",
             title="Natural Gas Price Trend (Henry Hub Futures)",
             labels={"Natural_Gas": "Price (USD/MMBtu)"},
             template="plotly_white", color_discrete_sequence=["#ff7f0e"])
fig.show()

### Viz 3: Construction Materials Trends (Steel & Cement Proxy)

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_market.index, y=df_market["Steel"], name="Steel (HRC Futures)", line=dict(color="#7f7f7f")), secondary_y=False)
fig.add_trace(go.Scatter(x=df_market.index, y=df_market["Cement_Proxy"], name="Cement Proxy (CX)", line=dict(color="#17becf")), secondary_y=True)
fig.update_layout(title_text="Construction Materials Price Trends (Steel vs. Cement Proxy)", template="plotly_white")
fig.update_yaxes(title_text="Steel Price (USD/Ton)", secondary_y=False)
fig.update_yaxes(title_text="Cement Proxy Price (USD/Share)", secondary_y=True)
fig.show()

### Viz 4: Daily Returns Correlation Matrix for Commodities

In [ ]:
returns = df_market[["Crude_Oil", "Natural_Gas", "Steel", "Cement_Proxy"]].pct_change().dropna()
corr = returns.corr()
fig = px.imshow(corr, text_auto=".2f", title="Commodities Daily Returns Correlation Matrix",
                color_continuous_scale="RdBu_r", zmin=-1, zmax=1)
fig.show()

### Viz 5: Normalized Foreign Exchange Rates (Base = 1.0)

In [ ]:
fx_cols = [c for c in df_market.columns if "USD_" in c]
df_fx_normalized = df_market[fx_cols] / df_market[fx_cols].iloc[0]
fig = px.line(df_fx_normalized.reset_index(), x="Date", y=fx_cols,
             title="Normalized Foreign Exchange Rates (Base = 1.0)",
             labels={"value": "Normalized Exchange Rate (USD/LCY)", "variable": "Currency Pair"},
             template="plotly_white")
fig.show()

### Viz 6: US Treasury Yield Curve Trends (SOFR Proxy)

In [ ]:
yield_cols = [c for c in df_market.columns if "US_Yield_" in c]
fig = px.line(df_market.reset_index(), x="Date", y=yield_cols,
             title="US Treasury Yield Curve Trends (SOFR Curve Tenors)",
             labels={"value": "Yield (%)", "variable": "Tenor"},
             template="plotly_white")
fig.show()

### Viz 7: US Treasury Yield Spread (10Y - 3M)

In [ ]:
df_market["Yield_Spread_10Y_3M"] = df_market["US_Yield_10Y"] - df_market["US_Yield_3M"]
fig = px.line(df_market.reset_index(), x="Date", y="Yield_Spread_10Y_3M",
              title="US Treasury Yield Spread (10Y - 3M) Inversion Indicator",
              labels={"Yield_Spread_10Y_3M": "Yield Spread (%)"},
              template="plotly_white", color_discrete_sequence=["#9467bd"])
fig.add_hline(y=0.0, line_dash="dash", line_color="red", annotation_text="Inversion Threshold")
fig.show()

### Viz 8: SOFR Nelson-Siegel Yield Curves for Select Dates

In [ ]:
dates_idx = [0, len(df_market)//2, -1]
dates = [df_market.index[i] for i in dates_idx]

tenors = np.linspace(0.25, 30.0, 100)
fig = go.Figure()

for d in dates:
    row = df_market.loc[d]
    b0, b1, b2, tau = row["sofr_ns_beta0"], row["sofr_ns_beta1"], row["sofr_ns_beta2"], row["sofr_ns_tau"]
    y = b0 + b1 * ((1 - np.exp(-tenors/tau)) / (tenors/tau)) + b2 * ((1 - np.exp(-tenors/tau)) / (tenors/tau) - np.exp(-tenors/tau))
    fig.add_trace(go.Scatter(x=tenors, y=y, mode="lines", name=f"SOFR Yield Curve - {d.strftime('%Y-%m-%d')}"))
    
fig.update_layout(title="Fitted Nelson-Siegel Yield Curves for SOFR (Select Dates)",
                  xaxis_title="Tenor (Years)", yaxis_title="Yield (%)", template="plotly_white")
fig.show()

### Viz 9: EURIBOR Nelson-Siegel Yield Curves for Select Dates

In [ ]:
fig = go.Figure()
for d in dates:
    row = df_market.loc[d]
    b0, b1, b2, tau = row["euribor_ns_beta0"], row["euribor_ns_beta1"], row["euribor_ns_beta2"], row["euribor_ns_tau"]
    y = b0 + b1 * ((1 - np.exp(-tenors/tau)) / (tenors/tau)) + b2 * ((1 - np.exp(-tenors/tau)) / (tenors/tau) - np.exp(-tenors/tau))
    fig.add_trace(go.Scatter(x=tenors, y=y, mode="lines", name=f"EURIBOR Yield Curve - {d.strftime('%Y-%m-%d')}"))
    
fig.update_layout(title="Fitted Nelson-Siegel Yield Curves for EURIBOR (Select Dates)",
                  xaxis_title="Tenor (Years)", yaxis_title="Yield (%)", template="plotly_white")
fig.show()

### Viz 10: Nelson-Siegel Beta Parameter Trends (SOFR)

In [ ]:
beta_cols = ["sofr_ns_beta0", "sofr_ns_beta1", "sofr_ns_beta2"]
fig = px.line(df_market.reset_index(), x="Date", y=beta_cols,
              title="Nelson-Siegel Parameter Trends over Time (SOFR)",
              labels={"value": "Parameter Value", "variable": "Beta Parameter"},
              template="plotly_white")
fig.show()

### Viz 11: Sovereign CDS Spreads by Credit Rating (Log Scale)

In [ ]:
cds_cols = [c for c in df_market.columns if "CDS_Spread_" in c]
fig = px.line(df_market.reset_index(), x="Date", y=cds_cols, log_y=True,
              title="Sovereign CDS Spreads by Credit Rating (Log Scale)",
              labels={"value": "CDS Spread (bps)", "variable": "Credit Rating"},
              template="plotly_white")
fig.show()

### Viz 12: Sovereign CDS Spread Distributions by Rating (Log Scale)

In [ ]:
df_cds_long = df_market[cds_cols].melt(var_name="Credit_Rating", value_name="CDS_Spread")
df_cds_long["Credit_Rating"] = df_cds_long["Credit_Rating"].str.replace("CDS_Spread_", "")
fig = px.violin(df_cds_long, x="Credit_Rating", y="CDS_Spread", color="Credit_Rating", box=True, log_y=True,
                title="Sovereign CDS Spread Distributions by Rating (Log Scale)",
                labels={"CDS_Spread": "CDS Spread (bps)", "Credit_Rating": "Rating Bracket"},
                template="plotly_white")
fig.show()

### Viz 13: Correlation Heatmap of Sovereign CDS Spreads

In [ ]:
corr_cds = df_market[cds_cols].corr()
corr_cds.columns = [c.replace("CDS_Spread_", "") for c in corr_cds.columns]
corr_cds.index = [c.replace("CDS_Spread_", "") for c in corr_cds.index]
fig = px.imshow(corr_cds, text_auto=".2f", title="Sovereign CDS Spreads Correlation Matrix",
                color_continuous_scale="RdBu_r", zmin=-1, zmax=1)
fig.show()

### Viz 14: GDP Growth vs. Inflation Rate (WDI Macro Data)

In [ ]:
fig = px.scatter(df_wb, x="gdp_growth", y="inflation", color="country_code",
                 hover_data=["year"],
                 title="GDP Growth vs. Inflation Rate (WDI Macro Data)",
                 labels={"gdp_growth": "GDP Growth (%)", "inflation": "Inflation Rate (%)"},
                 template="plotly_white")
fig.show()

### Viz 15: GDP Growth Distributions by Region (WDI)

In [ ]:
country_to_region = {c[0]: c[2] for c in wb_loader.COUNTRIES}
df_wb["region"] = df_wb["country_code"].map(country_to_region)
fig = px.box(df_wb.dropna(subset=["gdp_growth"]), x="region", y="gdp_growth", color="region",
             title="GDP Growth Distributions by Region (WDI)",
             labels={"gdp_growth": "GDP Growth (%)", "region": "Region"},
             template="plotly_white")
fig.show()

### Viz 16: Government Effectiveness vs. Regulatory Quality by Credit Rating

In [ ]:
country_to_rating = {c[0]: c[3] for c in wb_loader.COUNTRIES}
df_wb["sovereign_rating"] = df_wb["country_code"].map(country_to_rating)
rating_order = ["AAA", "AA", "A", "BBB+", "BBB", "BBB-", "BB+", "BB", "BB-", "B+", "B", "B-", "CCC+"]
rating_order = [r for r in rating_order if r in df_wb["sovereign_rating"].dropna().unique()]

fig = px.scatter(df_wb.dropna(subset=["government_effectiveness", "regulatory_quality"]), 
                 x="government_effectiveness", y="regulatory_quality", 
                 color="sovereign_rating", size="gdp_growth", hover_data=["country_code", "year"],
                 category_orders={"sovereign_rating": rating_order},
                 title="Government Effectiveness vs. Regulatory Quality by Credit Rating",
                 labels={"government_effectiveness": "Government Effectiveness Index", "regulatory_quality": "Regulatory Quality Index"},
                 template="plotly_white")
fig.show()